In [ ]:
import sys
import re
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../')
from utils.shap_utils import get_vals_feature
from utils.visualization_utils import visualize_int_row, visualize_interaction

# Get the data

In [ ]:
## Fill in your own settings here!!

# Insert your own WSI annotations here
wsi_annotations = ['Tumor', '--', 'Tissue edge: stroma > tumor', 'Tumor', 'Adipose > stroma', '--', 'Stroma > adipose', 'Adipose > stroma', 'Tumor: solid pattern, abundant cytoplasm', 'Tumor >> adipose', 'Stroma > tumor > immune cells', 'Tumor > stroma > immune cells', 'Normal ducts/lobules', 'Stroma > adipose', 'Edge stainings, blood vessels', 'Stroma >  immune cells']

# Data type
type = 'brca'

# Fold
fold = 2

# Name of experiment/model
exp_name = "DIMAFx"

# attention matrix type --> which of the 4 you want to look at 
# for the specific Wx: take self_attn_wsi
# for the specific Rx: take self_attn_rna
# for the shared Wx: take cross_attn_rna_wsi
# for the shared Rx: take cross_attn_wsi_rna
attn_type = 'cross_attn_rna_wsi'

# Multimodal feature  --> shared W8
ft_name1 = "W8"

In [ ]:
# Loading pathways names and initializing cluster identities
hallmarks = pd.read_csv('../data/data_files/hallmarks_signatures.csv')
hallmarks = np.array([' '.join(x[9:].split('_')) for x in hallmarks.columns])
hallmarks_ids = np.array([f'R{i}' for i, item in enumerate (hallmarks)])
hallmarks_names = np.array([f'R{i}: {item}' for i, item in enumerate (hallmarks)])

cluster_ids = np.array([f'W{i}' for i, item in enumerate (wsi_annotations)])
cluster_names = np.array([f'W{i}: {item}' for i, item in enumerate (wsi_annotations)])

# Obtain attention matrix
data_results_path = f"../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/attention/learned_attention_matrices_test.pkl"
data_attn = pickle.load(open(data_results_path, 'rb'))     

# Find most important interactions

In [ ]:
visualize_int_row(data_attn, attn_type, cluster_names, hallmarks_names, search_name=ft_name1)

# Zoom into specific found interaction

In [ ]:
# feature with highest interaction
ft_name2 = "R13"

In [ ]:
# Unimodal shap
shap_results_fold_dir = f'../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/shap/modal/shap_all_test.pkl'
shap_dict = pickle.load(open(shap_results_fold_dir, 'rb'))   
shap_feature1 = get_vals_feature(shap_dict, ft_name1)
shap_feature2 = get_vals_feature(shap_dict, ft_name2)

# Attention vals
log2_risk_scores = np.log2(data_attn["Risk scores"])

In [ ]:
visualize_interaction(shap_feature1, shap_feature2, color_val=log2_risk_scores, row_name=f'SHAP of {ft_name1}', col_name=f'SHAP of {ft_name2}', color_name='Log2\nRisk', title=f"Interaction: {ft_name1} vs {ft_name2}")